# Europe PMC Tools

![Europe PMC Tools](https://proto-bio.github.io/proto-assets/images/tool/europepmc/hero.png)

This notebook demonstrates the two Europe PMC tools. `run_europepmc_search` finds papers and preprints in Europe PMC by free text or field query, and `run_europepmc_fulltext` reads an open-access article as plain-text sections. Together they take you from a design question to the exact passage, table row, or figure caption that answers it.

In [ ]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("europepmc")
display_overview("europepmc")
display_docs_section("europepmc", "Background")

## Available tools

In [ ]:
display_available_tools("europepmc")

### `run_europepmc_search`

`run_europepmc_search` ranks Europe PMC records for a query written in free text or Europe PMC syntax. Each hit carries its PMID, PMCID, and DOI, citation details, whether it is open access, and its abstract. Here we look for open-access papers on de novo designed binders that report binding affinities.

In [ ]:
from proto_tools import (
    EuropePMCSearchConfig,
    EuropePMCSearchInput,
    run_europepmc_search,
)

In [ ]:
# Display docs
display_api_reference("europepmc", "input", "run_europepmc_search")

# Search open-access papers on de novo binder design that mention affinity
inputs = EuropePMCSearchInput(queries='"de novo" "binder design" affinity OPEN_ACCESS:y')

In [ ]:
# Display config docs
display_api_reference("europepmc", "config", "run_europepmc_search")

# Return the top 5 hits with abstracts | see docs above for all fields
config = EuropePMCSearchConfig(max_results=5)

In [ ]:
# Run the search
search_output = run_europepmc_search(inputs, config)

In [ ]:
# Display output docs
display_api_reference("europepmc", "output", "run_europepmc_search")

# Inspect the ranked hits
result = search_output.results[0]
print(f"{result.total_count} matching records")
for hit in result.hits:
    print(f"{hit.pmcid or hit.id}  ({hit.year})  OA={hit.is_open_access}  {hit.title[:80]}")

#### Preprints and Field Queries

Field codes narrow a search more reliably than extra words. `SRC:PPR` keeps only preprints, including bioRxiv and medRxiv, which makes this the keyword search for preprints. A `DOI:` query resolves a paper you already know to its PMID and PMCID.

In [ ]:
preprint_output = run_europepmc_search(
    EuropePMCSearchInput(queries=['TITLE:"protein binder" AND SRC:PPR', 'DOI:"10.1038/s41586-023-06415-8"']),
    EuropePMCSearchConfig(max_results=3, include_abstracts=False),
)

for result in preprint_output.results:
    print(f"\n{result.query}  ({result.total_count} records)")
    for hit in result.hits:
        print(f"  [{hit.source}] {hit.journal}  doi={hit.doi}  pmcid={hit.pmcid}  {hit.title[:60]}")

### `run_europepmc_fulltext`

`run_europepmc_fulltext` fetches an open-access article's JATS XML and flattens it into sections: the title, abstracts, body and back-matter sections under their heading path, figure captions, and tables with cells separated by `|`. Here we read the RFdiffusion paper resolved by DOI above.

In [ ]:
from proto_tools import (
    EuropePMCFullTextConfig,
    EuropePMCFullTextInput,
    run_europepmc_fulltext,
)

In [ ]:
# Display docs
display_api_reference("europepmc", "input", "run_europepmc_fulltext")

# Read the RFdiffusion paper by its PMCID
fulltext_inputs = EuropePMCFullTextInput(pmcids=["PMC10468394"])

In [ ]:
# Display config docs
display_api_reference("europepmc", "config", "run_europepmc_fulltext")

# Use default config | see docs above for all fields
fulltext_config = EuropePMCFullTextConfig()

In [ ]:
# Fetch the full text
fulltext_output = run_europepmc_fulltext(fulltext_inputs, fulltext_config)

In [ ]:
# Display output docs
display_api_reference("europepmc", "output", "run_europepmc_fulltext")

# Inspect the document's sections
document = fulltext_output.documents[0]
print(f"{document.title}\nPMID {document.pmid}  DOI {document.doi}  license {document.license}\n")
for section in document.sections[:12]:
    print(f"[{section.kind}] {section.heading}: {len(section.text)} chars")

#### Finding a Supporting Passage

Because every section is plain text, a regular expression finds the sentence that reports a value. Here we pull each sentence that states a binding affinity in nanomolar, with the heading it sits under, so it can be quoted and cited verbatim.

In [ ]:
import re

# Split on sentence ends (a period followed by a capital letter), then keep affinity statements in nM
sentence_end = re.compile(r"(?<=[.!?])\s+(?=[A-Z])")
affinity = re.compile(r"\b(?:K_?[dD]|affinit\w*)\b.*\bnM\b")
for section in document.sections:
    for sentence in sentence_end.split(section.text.replace("\n", " ")):
        if affinity.search(sentence):
            print(f"[{section.heading}] {sentence[:240]}\n")

## Export Results

Search results and documents can be saved to JSON, and documents also to plain text for searching with other tools. Each article keeps its own copyright and license, so this example writes to a temporary directory rather than committing retrieved text.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as output_dir:
    search_output.export("europepmc_search", export_path=output_dir, file_format="csv")
    fulltext_output.export("europepmc_fulltext", export_path=output_dir, file_format="txt")
    print(f"Exported {sorted(path.name for path in Path(output_dir).iterdir())}")